In [0]:
%pip install feature-engine lightgbm
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
from sklearn import model_selection, metrics

CUTOFF = 0.29

# Champion model from the MLflow registry
model = mlflow.sklearn.load_model(
    "models:/workspace.default.churn_lightgbm_balanced_tuned@champion"
)

# Historical customers with known outcome
df = spark.table("workspace.default.raw_abt2_churn").toPandas()
feats = list(model.feature_names_in_)
y = df["TargetChurn"]

# Same hold-out split used in training: 20% test, seed 42, stratified
_, idx_test = model_selection.train_test_split(
    df.index, random_state=42, stratify=y, test_size=0.2
)
test = df.loc[idx_test]

# Real outcome + model probability, side by side
out = pd.DataFrame({
    "customerID": test["customerID"].values if "customerID" in test.columns else test.index.astype(str),
    "TargetChurn": test["TargetChurn"].astype(int).values,
    "ProbaChurn": model.predict_proba(test[feats])[:, 1],
})

# Sanity check: expect AUC ~0.844, recall ~0.77, precision ~0.55 at 0.29
pred = (out.ProbaChurn >= CUTOFF).astype(int)
print("n test:", len(out))
print("AUC:", round(metrics.roc_auc_score(out.TargetChurn, out.ProbaChurn), 4))
print("Recall:", round(metrics.recall_score(out.TargetChurn, pred), 4),
      "| Precision:", round(metrics.precision_score(out.TargetChurn, pred), 4),
      "| F1:", round(metrics.f1_score(out.TargetChurn, pred), 4))

# Save for the dashboard (overwrite: always rebuilt from scratch)
spark.createDataFrame(out).write.mode("overwrite").saveAsTable(
    "workspace.default.churn_validation_scores"
)

In [0]:
%sql
SELECT *
FROM workspace.default.churn_scores c
JOIN workspace.default.output_churn o USING (customerID);

In [0]:
%sql
SELECT COUNT(*) AS n,
       SUM(CASE WHEN ABS(c.ProbaChurn - o.ProbaChurn) > 1e-9 THEN 1 ELSE 0 END) AS prob_diferente,
       AVG(CASE WHEN c.prediction = 'Yes' THEN 1 ELSE 0 END) AS pct_yes_churn_scores
FROM workspace.default.churn_scores c
JOIN workspace.default.output_churn o USING (customerID);